In [2]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    print("GPU:", gpu)

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
GPU: PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')


In [3]:
import subprocess

print(subprocess.check_output(["nvidia-smi"]).decode())

Tue Sep 15 17:35:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import psutil

ram = psutil.virtual_memory()

print("Total RAM:", round(ram.total / (1024**3), 2), "GB")
print("Available RAM:", round(ram.available / (1024**3), 2), "GB")
print("Used RAM:", round(ram.used / (1024**3), 2), "GB")
print("RAM Usage:", ram.percent, "%")

Total RAM: 31.35 GB
Available RAM: 29.58 GB
Used RAM: 1.35 GB
RAM Usage: 5.6 %


In [5]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/gou14226/m5-features-event-snap/features_event_snap.parquet


In [6]:
import pyarrow.parquet as pq
import os

DATA_PATH = "/kaggle/input/datasets/gou14226/m5-features-event-snap/features_event_snap.parquet"

parquet_file = pq.ParquetFile(DATA_PATH)

print("File exists:", os.path.exists(DATA_PATH))
print("File size:", round(os.path.getsize(DATA_PATH) / (1024**3), 2), "GB")
print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.metadata.num_columns)
print("Row groups:", parquet_file.num_row_groups)

print("\nColumns:")
print(parquet_file.schema_arrow.names)

File exists: True
File size: 0.44 GB
Rows: 58327370
Columns: 42
Row groups: 61

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'price_change_1', 'price_change_pct_1', 'price_relative_7', 'is_event_day', 'event_count', 'snap_active']


In [7]:
# ============================================================
# GRU CONFIGURATION
# ============================================================

SEQUENCE_LENGTH = 28
FORECAST_HORIZON = 1
TARGET_COLUMN = "sales"

FEATURE_COLUMNS = [
    "sales",
    "sell_price",
    "price_available",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7",
    "is_event_day",
    "event_count",
    "snap_active"
]

DATA_PATH = "/kaggle/input/datasets/gou14226/m5-features-event-snap/features_event_snap.parquet"

TRAIN_END_DATE = "2016-03-27"
VALIDATION_START_DATE = "2016-03-28"
VALIDATION_END_DATE = "2016-04-24"

BATCH_SIZE = 64
EPOCHS = 10
TRAIN_STEPS = 5000
VALIDATION_STEPS = 500

TEST_SERIES_LIMIT = 1

print("Sequence length:", SEQUENCE_LENGTH)
print("Number of features:", len(FEATURE_COLUMNS))
print("Batch size:", BATCH_SIZE)
print("Train end date:", TRAIN_END_DATE)
print("Validation period:", VALIDATION_START_DATE, "to", VALIDATION_END_DATE)

Sequence length: 28
Number of features: 22
Batch size: 64
Train end date: 2016-03-27
Validation period: 2016-03-28 to 2016-04-24


In [8]:
# ============================================================
# FEATURE PREPARATION
# ============================================================

def prepare_features(df, feature_columns):
    df = df.copy()

    price_features = [
        "sell_price",
        "price_change_1",
        "price_change_pct_1",
        "price_relative_7"
    ]

    history_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    # Missing price-related features
    # are intentionally represented as 0 for model input.
    for col in price_features:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Missing history/rolling features
    # are also represented as 0 for model input.
    for col in history_features:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    X = df[feature_columns].to_numpy(dtype=np.float64)

    return X

print("prepare_features() function created successfully.")

prepare_features() function created successfully.


In [9]:
# ============================================================
# TRAINING-ONLY FEATURE SCALER
# ============================================================
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.preprocessing import StandardScaler

parquet_file = pq.ParquetFile(DATA_PATH)

scaler = StandardScaler()

TRAINING_ROWS = 0

for rg_idx in range(parquet_file.num_row_groups):

    table = parquet_file.read_row_group(
        rg_idx,
        columns=FEATURE_COLUMNS + ["date"]
    )

    df_rg = table.to_pandas()

    # Keep only training-period rows
    train_mask = df_rg["date"] <= pd.Timestamp(TRAIN_END_DATE)
    train_df = df_rg.loc[train_mask]

    if len(train_df) > 0:
        X_train_rg = prepare_features(
            train_df,
            FEATURE_COLUMNS
        )

        scaler.partial_fit(X_train_rg)
        TRAINING_ROWS += len(train_df)

    if (rg_idx + 1) % 10 == 0 or rg_idx == parquet_file.num_row_groups - 1:
        print(
            f"Processed row group {rg_idx + 1}/{parquet_file.num_row_groups} "
            f"| Training rows: {TRAINING_ROWS:,}"
        )

print("\nScaler fitting complete.")
print("Total training rows:", f"{TRAINING_ROWS:,}")
print("Number of features:", len(scaler.mean_))
print("First 5 means:", scaler.mean_[:5])
print("First 5 stds:", scaler.scale_[:5])

Processed row group 10/61 | Training rows: 9,425,000
Processed row group 20/61 | Training rows: 18,850,000
Processed row group 30/61 | Training rows: 28,275,000
Processed row group 40/61 | Training rows: 37,700,000
Processed row group 50/61 | Training rows: 47,125,000
Processed row group 60/61 | Training rows: 56,550,000
Processed row group 61/61 | Training rows: 57,473,650

Scaler fitting complete.
Total training rows: 57,473,650
Number of features: 22
First 5 means: [ 1.12245843  3.4636664   0.7859991  15.71458886 26.0397878 ]
First 5 stds: [ 3.87698193  3.51547202  0.41012744  8.79354866 15.17225613]


In [10]:
# ============================================================
# SCALER VERIFICATION
# ============================================================

test_table = parquet_file.read_row_group(
    0,
    columns=FEATURE_COLUMNS
)

test_df = test_table.to_pandas()

X_test_raw = prepare_features(
    test_df,
    FEATURE_COLUMNS
)

X_test_scaled = scaler.transform(X_test_raw)

print("Raw shape:", X_test_raw.shape)
print("Scaled shape:", X_test_scaled.shape)

print("NaN count:", np.isnan(X_test_scaled).sum())
print("Inf count:", np.isinf(X_test_scaled).sum())

print("\nFirst 10 scaled values:")
print(X_test_scaled[0, :10])

Raw shape: (1048576, 22)
Scaled shape: (1048576, 22)
NaN count: 0
Inf count: 0

First 10 scaled values:
[-0.28951861 -0.98526354 -1.91647527  1.5108134  -1.45263747 -1.41595634
 -1.29299269  1.57820808 -0.2894102  -0.28867355]


In [12]:
# ============================================================
# SEQUENCE GENERATOR
# ============================================================

def create_sequences(
    df,
    feature_columns,
    target_column,
    scaler,
    sequence_length=28
):
    df = df.sort_values("date").reset_index(drop=True)

    X_raw = prepare_features(
        df,
        feature_columns
    )

    X_scaled = scaler.transform(X_raw)

    y = df[target_column].to_numpy(dtype=np.float32)

    X_sequences = []
    y_sequences = []

    for i in range(sequence_length, len(df)):
        X_sequences.append(
            X_scaled[i-sequence_length:i]
        )
        y_sequences.append(
            y[i]
        )

    if len(X_sequences) == 0:
        return (
            np.empty(
                (0, sequence_length, len(feature_columns)),
                dtype=np.float32
            ),
            np.empty(
                (0,),
                dtype=np.float32
            )
        )

    return (
        np.asarray(X_sequences, dtype=np.float32),
        np.asarray(y_sequences, dtype=np.float32)
    )

print("create_sequences() function created successfully.")

create_sequences() function created successfully.


In [13]:
# Select one product-store series for testing
TEST_ITEM = "HOBBIES_1_001"
TEST_STORE = "CA_1"

# Read only required columns
test_table = parquet_file.read_row_group(
    0,
    columns=FEATURE_COLUMNS + ["date", "item_id", "store_id"]
)

test_df = test_table.to_pandas()

# Filter selected series
test_series = test_df[
    (test_df["item_id"] == TEST_ITEM) &
    (test_df["store_id"] == TEST_STORE)
].copy()

print("Test series:", TEST_ITEM, "/", TEST_STORE)
print("Rows:", len(test_series))
print("Date range:", test_series["date"].min(), "to", test_series["date"].max())

# Create sequences
X_test_seq, y_test_seq = create_sequences(
    test_series,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    scaler,
    sequence_length=SEQUENCE_LENGTH
)

print("\nSequence test results:")
print("X shape:", X_test_seq.shape)
print("y shape:", y_test_seq.shape)

print("\nNaN count in X:", np.isnan(X_test_seq).sum())
print("Inf count in X:", np.isinf(X_test_seq).sum())
print("NaN count in y:", np.isnan(y_test_seq).sum())
print("Inf count in y:", np.isinf(y_test_seq).sum())

print("\nFirst target values:")
print(y_test_seq[:10])

Test series: HOBBIES_1_001 / CA_1
Rows: 1049
Date range: 2011-01-29 00:00:00 to 2013-12-12 00:00:00

Sequence test results:
X shape: (1021, 28, 22)
y shape: (1021,)

NaN count in X: 0
Inf count in X: 0
NaN count in y: 0
Inf count in y: 0

First target values:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [14]:
# Check the date range of the complete dataset
date_table = parquet_file.read(
    columns=["date"]
).to_pandas()

print("Global date range:")
print(date_table["date"].min(), "to", date_table["date"].max())

print("\nTrain period:")
print(TRAIN_END_DATE)

print("\nValidation period:")
print(VALIDATION_START_DATE, "to", VALIDATION_END_DATE)

Global date range:
2011-01-29 00:00:00 to 2016-04-24 00:00:00

Train period:
2016-03-27

Validation period:
2016-03-28 to 2016-04-24


In [16]:
# Restore imports and configuration after runtime restart

import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tensorflow as tf

from sklearn.preprocessing import StandardScaler

SEQUENCE_LENGTH = 28
FORECAST_HORIZON = 1
TARGET_COLUMN = "sales"

FEATURE_COLUMNS = [
    "sales",
    "sell_price",
    "price_available",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7",
    "is_event_day",
    "event_count",
    "snap_active"
]

DATA_PATH = (
    "/kaggle/input/datasets/gou14226/"
    "m5-features-event-snap/features_event_snap.parquet"
)

TRAIN_END_DATE = "2016-03-27"
VALIDATION_START_DATE = "2016-03-28"
VALIDATION_END_DATE = "2016-04-24"

BATCH_SIZE = 64
EPOCHS = 10
TRAIN_STEPS = 5000
VALIDATION_STEPS = 500

parquet_file = pq.ParquetFile(DATA_PATH)

print("Configuration restored successfully.")
print("Dataset exists:", os.path.exists(DATA_PATH))
print("Row groups:", parquet_file.num_row_groups)
print("Number of features:", len(FEATURE_COLUMNS))
print("Sequence length:", SEQUENCE_LENGTH)
print("Batch size:", BATCH_SIZE)

Configuration restored successfully.
Dataset exists: True
Row groups: 61
Number of features: 22
Sequence length: 28
Batch size: 64


In [17]:
def prepare_features(df, feature_columns):
    df = df.copy()

    # Price-related features:
    # Missing price means price was unavailable.
    # We keep the raw dataset unchanged and use 0 only for model input.
    price_features = [
        "sell_price",
        "price_change_1",
        "price_change_pct_1",
        "price_relative_7"
    ]

    # History/rolling features:
    # Initial days naturally have missing historical values.
    history_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    for col in price_features:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    for col in history_features:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    X = df[feature_columns].to_numpy(dtype=np.float64)

    return X


print("prepare_features() function restored successfully.")

prepare_features() function restored successfully.


In [18]:
# Fit StandardScaler using training data only
# Process one Parquet row group at a time to avoid high RAM usage.

scaler = StandardScaler()
TRAINING_ROWS = 0

for rg_idx in range(parquet_file.num_row_groups):

    table = parquet_file.read_row_group(
        rg_idx,
        columns=FEATURE_COLUMNS + ["date"]
    )

    df_rg = table.to_pandas()

    # Only training-period rows
    train_mask = df_rg["date"] <= pd.Timestamp(TRAIN_END_DATE)
    train_df = df_rg.loc[train_mask]

    if len(train_df) > 0:
        X_train_rg = prepare_features(
            train_df,
            FEATURE_COLUMNS
        )

        scaler.partial_fit(X_train_rg)
        TRAINING_ROWS += len(train_df)

    # Progress update
    if (rg_idx + 1) % 10 == 0 or rg_idx == parquet_file.num_row_groups - 1:
        print(
            f"Processed row group "
            f"{rg_idx + 1}/{parquet_file.num_row_groups} "
            f"| Training rows: {TRAINING_ROWS:,}"
        )

print("\nScaler fitting complete.")
print("Total training rows:", f"{TRAINING_ROWS:,}")
print("Number of features:", len(scaler.mean_))

print("\nFirst 5 means:")
print(scaler.mean_[:5])

print("\nFirst 5 stds:")
print(scaler.scale_[:5])

Processed row group 10/61 | Training rows: 9,425,000
Processed row group 20/61 | Training rows: 18,850,000
Processed row group 30/61 | Training rows: 28,275,000
Processed row group 40/61 | Training rows: 37,700,000
Processed row group 50/61 | Training rows: 47,125,000
Processed row group 60/61 | Training rows: 56,550,000
Processed row group 61/61 | Training rows: 57,473,650

Scaler fitting complete.
Total training rows: 57,473,650
Number of features: 22

First 5 means:
[ 1.12245843  3.4636664   0.7859991  15.71458886 26.0397878 ]

First 5 stds:
[ 3.87698193  3.51547202  0.41012744  8.79354866 15.17225613]


In [19]:
def generate_sequences_from_row_group(
    df,
    feature_columns,
    target_column,
    scaler,
    sequence_length,
    start_date,
    end_date
):
    """
    Generate sequences from a single Parquet row group.

    Only the requested date range is returned.
    No complete dataset is accumulated in memory.
    """

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    # Sort within the row group
    df = df.sort_values(
        ["item_id", "store_id", "date"]
    ).reset_index(drop=True)

    X_sequences = []
    y_sequences = []

    # Process one item-store series at a time
    for (item_id, store_id), group in df.groupby(
        ["item_id", "store_id"],
        sort=False
    ):
        group = group.sort_values("date").reset_index(drop=True)

        if len(group) <= sequence_length:
            continue

        # Prepare and scale features
        X_raw = prepare_features(
            group,
            feature_columns
        )

        X_scaled = scaler.transform(X_raw)

        y = group[target_column].to_numpy(
            dtype=np.float32
        )

        dates = group["date"]

        # Only targets inside requested period
        valid_indices = np.where(
            (dates >= start_date) &
            (dates <= end_date)
        )[0]

        for i in valid_indices:

            # Need 28 previous days
            if i < sequence_length:
                continue

            X_sequences.append(
                X_scaled[i-sequence_length:i]
            )

            y_sequences.append(y[i])

    if len(X_sequences) == 0:
        return (
            np.empty(
                (0, sequence_length, len(feature_columns)),
                dtype=np.float32
            ),
            np.empty(
                (0,),
                dtype=np.float32
            )
        )

    return (
        np.asarray(X_sequences, dtype=np.float32),
        np.asarray(y_sequences, dtype=np.float32)
    )


print("Memory-safe sequence generator function created successfully.")

Memory-safe sequence generator function created successfully.


In [20]:
# Read ONLY the first Parquet row group
test_table = parquet_file.read_row_group(
    0,
    columns=FEATURE_COLUMNS + [
        "date",
        "item_id",
        "store_id"
    ]
)

test_rg = test_table.to_pandas()

print("Row group shape:", test_rg.shape)
print(
    "Row group date range:",
    test_rg["date"].min(),
    "to",
    test_rg["date"].max()
)

# Generate sequences from this single row group
X_rg_test, y_rg_test = generate_sequences_from_row_group(
    df=test_rg,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    scaler=scaler,
    sequence_length=SEQUENCE_LENGTH,
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE
)

print("\nGenerated sequences:")
print("X shape:", X_rg_test.shape)
print("y shape:", y_rg_test.shape)

print("\nData quality:")
print("NaN in X:", np.isnan(X_rg_test).sum())
print("Inf in X:", np.isinf(X_rg_test).sum())
print("NaN in y:", np.isnan(y_rg_test).sum())
print("Inf in y:", np.isinf(y_rg_test).sum())

Row group shape: (1048576, 25)
Row group date range: 2011-01-29 00:00:00 to 2013-12-12 00:00:00

Generated sequences:
X shape: (1020576, 28, 22)
y shape: (1020576,)

Data quality:
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0


In [21]:
# Release the large single-row-group test arrays
del X_rg_test
del y_rg_test
del test_rg
del test_table

import gc
gc.collect()

print("Single row-group test data released from memory.")
print(
    "Available RAM (GB):",
    round(
        __import__("psutil").virtual_memory().available / (1024**3),
        2
    )
)

Single row-group test data released from memory.
Available RAM (GB): 27.27


In [22]:
def generate_batch_from_row_group(
    df,
    feature_columns,
    target_column,
    scaler,
    sequence_length,
    start_date,
    end_date,
    batch_size=64
):
    """
    Generate one batch of time-series sequences from a single row group.

    Sequences are created series-by-series and returned as soon
    as batch_size samples are available.
    """

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    df = df.sort_values(
        ["item_id", "store_id", "date"]
    ).reset_index(drop=True)

    X_batch = []
    y_batch = []

    for (item_id, store_id), group in df.groupby(
        ["item_id", "store_id"],
        sort=False
    ):
        group = group.sort_values("date").reset_index(drop=True)

        if len(group) <= sequence_length:
            continue

        X_raw = prepare_features(
            group,
            feature_columns
        )

        X_scaled = scaler.transform(X_raw)

        y = group[target_column].to_numpy(
            dtype=np.float32
        )

        dates = group["date"]

        valid_indices = np.where(
            (dates >= start_date) &
            (dates <= end_date)
        )[0]

        for i in valid_indices:

            if i < sequence_length:
                continue

            X_batch.append(
                X_scaled[i-sequence_length:i]
            )

            y_batch.append(y[i])

            if len(X_batch) == batch_size:
                yield (
                    np.asarray(X_batch, dtype=np.float32),
                    np.asarray(y_batch, dtype=np.float32)
                )

                X_batch = []
                y_batch = []

    # Return final partial batch if any
    if len(X_batch) > 0:
        yield (
            np.asarray(X_batch, dtype=np.float32),
            np.asarray(y_batch, dtype=np.float32)
        )


print("Batch-wise sequence generator created successfully.")

Batch-wise sequence generator created successfully.


In [23]:
# Read only the first Parquet row group
test_table = parquet_file.read_row_group(
    0,
    columns=FEATURE_COLUMNS + [
        "date",
        "item_id",
        "store_id"
    ]
)

test_rg = test_table.to_pandas()

# Generate only ONE batch
batch_generator = generate_batch_from_row_group(
    df=test_rg,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    scaler=scaler,
    sequence_length=SEQUENCE_LENGTH,
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    batch_size=BATCH_SIZE
)

X_batch, y_batch = next(batch_generator)

print("Batch generated successfully.")
print("X shape:", X_batch.shape)
print("y shape:", y_batch.shape)

print("\nData types:")
print("X dtype:", X_batch.dtype)
print("y dtype:", y_batch.dtype)

print("\nData quality:")
print("NaN in X:", np.isnan(X_batch).sum())
print("Inf in X:", np.isinf(X_batch).sum())
print("NaN in y:", np.isnan(y_batch).sum())
print("Inf in y:", np.isinf(y_batch).sum())

print("\nFirst 10 target values:")
print(y_batch[:10])

Batch generated successfully.
X shape: (64, 28, 22)
y shape: (64,)

Data types:
X dtype: float32
y dtype: float32

Data quality:
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

First 10 target values:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [24]:
# Release test data from memory
del X_batch
del y_batch
del batch_generator
del test_rg
del test_table

import gc
gc.collect()

print("Test batch memory released.")

import psutil
print(
    "Available RAM (GB):",
    round(psutil.virtual_memory().available / (1024**3), 2)
)

Test batch memory released.
Available RAM (GB): 27.42


In [25]:
def stream_batches_from_parquet(
    parquet_path,
    feature_columns,
    target_column,
    scaler,
    sequence_length,
    start_date,
    end_date,
    batch_size=64
):
    """
    Stream training/validation batches from Parquet
    without loading the complete dataset into RAM.
    """

    pf = pq.ParquetFile(parquet_path)

    required_columns = list(
        dict.fromkeys(
            feature_columns
            + [target_column, "date", "item_id", "store_id"]
        )
    )

    for rg_idx in range(pf.num_row_groups):

        table = pf.read_row_group(
            rg_idx,
            columns=required_columns
        )

        df_rg = table.to_pandas()

        # Skip row groups that are completely outside the target period
        if df_rg["date"].min() > pd.Timestamp(end_date):
            del df_rg
            del table
            continue

        if df_rg["date"].max() < pd.Timestamp(start_date):
            del df_rg
            del table
            continue

        batch_generator = generate_batch_from_row_group(
            df=df_rg,
            feature_columns=feature_columns,
            target_column=target_column,
            scaler=scaler,
            sequence_length=sequence_length,
            start_date=start_date,
            end_date=end_date,
            batch_size=batch_size
        )

        for X_batch, y_batch in batch_generator:
            yield X_batch, y_batch

        del batch_generator
        del df_rg
        del table


print("Parquet streaming batch generator created successfully.")

Parquet streaming batch generator created successfully.


In [26]:
# Test the streaming generator with only 3 batches

stream = stream_batches_from_parquet(
    parquet_path=DATA_PATH,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    scaler=scaler,
    sequence_length=SEQUENCE_LENGTH,
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    batch_size=BATCH_SIZE
)

for batch_num in range(3):
    X_batch, y_batch = next(stream)

    print(f"\nBatch {batch_num + 1}:")
    print("X shape:", X_batch.shape)
    print("y shape:", y_batch.shape)
    print("X dtype:", X_batch.dtype)
    print("y dtype:", y_batch.dtype)
    print("NaN in X:", np.isnan(X_batch).sum())
    print("Inf in X:", np.isinf(X_batch).sum())
    print("NaN in y:", np.isnan(y_batch).sum())
    print("Inf in y:", np.isinf(y_batch).sum())

print("\n3-batch streaming test completed.")


Batch 1:
X shape: (64, 28, 22)
y shape: (64,)
X dtype: float32
y dtype: float32
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

Batch 2:
X shape: (64, 28, 22)
y shape: (64,)
X dtype: float32
y dtype: float32
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

Batch 3:
X shape: (64, 28, 22)
y shape: (64,)
X dtype: float32
y dtype: float32
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

3-batch streaming test completed.


In [27]:
def stream_series_from_parquet(
    parquet_path,
    feature_columns,
    target_column,
    start_date,
    end_date,
    sequence_length=28
):
    """
    Stream item-store time series from Parquet while preserving
    continuity across row-group boundaries.

    Only one row group is held in memory at a time.
    """

    pf = pq.ParquetFile(parquet_path)

    required_columns = list(
        dict.fromkeys(
            feature_columns
            + [
                target_column,
                "date",
                "item_id",
                "store_id"
            ]
        )
    )

    current_key = None
    series_buffer = []

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    for rg_idx in range(pf.num_row_groups):

        table = pf.read_row_group(
            rg_idx,
            columns=required_columns
        )

        df_rg = table.to_pandas()

        df_rg = df_rg.sort_values(
            ["item_id", "store_id", "date"]
        ).reset_index(drop=True)

        for (item_id, store_id), group in df_rg.groupby(
            ["item_id", "store_id"],
            sort=False
        ):

            group = group.sort_values("date").reset_index(drop=True)
            key = (item_id, store_id)

            # New series
            if current_key is None:
                current_key = key

            # Series changed: yield previous series
            if key != current_key:

                if len(series_buffer) > sequence_length:
                    series_df = pd.concat(
                        series_buffer,
                        ignore_index=True
                    )

                    yield current_key, series_df

                series_buffer = []
                current_key = key

            series_buffer.append(group)

        del df_rg
        del table

    # Yield final series
    if current_key is not None and len(series_buffer) > 0:

        series_df = pd.concat(
            series_buffer,
            ignore_index=True
        )

        if len(series_df) > sequence_length:
            yield current_key, series_df


print("Cross-row-group series streaming function created successfully.")

Cross-row-group series streaming function created successfully.


In [28]:
TEST_ITEM = "HOBBIES_1_001"
TEST_STORE = "CA_1"

series_found = False

for (item_id, store_id), series_df in stream_series_from_parquet(
    parquet_path=DATA_PATH,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    sequence_length=SEQUENCE_LENGTH
):

    if item_id == TEST_ITEM and store_id == TEST_STORE:

        series_found = True

        print("Series found:", item_id, "/", store_id)
        print("Rows:", len(series_df))
        print(
            "Date range:",
            series_df["date"].min(),
            "to",
            series_df["date"].max()
        )

        print("\nFirst 5 dates:")
        print(series_df["date"].head().tolist())

        print("\nLast 5 dates:")
        print(series_df["date"].tail().tolist())

        # Check date continuity
        date_diff = series_df["date"].diff().dropna()
        expected_gap = pd.Timedelta(days=1)

        print(
            "\nNon-1-day gaps:",
            (date_diff != expected_gap).sum()
        )

        break

if not series_found:
    print("Test series was not found.")

Test series was not found.


In [29]:
# Inspect the first few row groups to understand
# how item-store series are physically organized.

for rg_idx in range(3):

    table = parquet_file.read_row_group(
        rg_idx,
        columns=["item_id", "store_id", "date"]
    )

    df_rg = table.to_pandas()

    print(f"\nRow group {rg_idx}:")
    print("Rows:", len(df_rg))
    print(
        "First series:",
        df_rg.iloc[0]["item_id"],
        "/",
        df_rg.iloc[0]["store_id"]
    )
    print(
        "Last series:",
        df_rg.iloc[-1]["item_id"],
        "/",
        df_rg.iloc[-1]["store_id"]
    )

    print(
        "Unique item-store pairs:",
        df_rg[["item_id", "store_id"]].drop_duplicates().shape[0]
    )

    del df_rg
    del table

print("\nRow-group organization inspection completed.")


Row group 0:
Rows: 1048576
First series: HOBBIES_1_001 / CA_1
Last series: HOUSEHOLD_1_444 / CA_1
Unique item-store pairs: 1000

Row group 1:
Rows: 864424
First series: HOBBIES_1_001 / CA_1
Last series: HOUSEHOLD_1_444 / CA_1
Unique item-store pairs: 1000

Row group 2:
Rows: 1048576
First series: FOODS_1_001 / CA_1
Last series: HOUSEHOLD_2_516 / CA_1
Unique item-store pairs: 1000

Row-group organization inspection completed.


In [30]:
# Check how the same series continues across row-group 0 and row-group 1

cols = ["item_id", "store_id", "date", "sales"]

rg0 = parquet_file.read_row_group(
    0,
    columns=cols
).to_pandas()

rg1 = parquet_file.read_row_group(
    1,
    columns=cols
).to_pandas()

test_key = ("HOBBIES_1_001", "CA_1")

series0 = rg0[
    (rg0["item_id"] == test_key[0]) &
    (rg0["store_id"] == test_key[1])
].sort_values("date")

series1 = rg1[
    (rg1["item_id"] == test_key[0]) &
    (rg1["store_id"] == test_key[1])
].sort_values("date")

print("Test series:", test_key)

print("\nRow group 0:")
print("Rows:", len(series0))
print("Date range:", series0["date"].min(), "to", series0["date"].max())

print("\nRow group 1:")
print("Rows:", len(series1))
print("Date range:", series1["date"].min(), "to", series1["date"].max())

print("\nBoundary dates:")
print("RG0 last 3 dates:")
print(series0["date"].tail(3).tolist())

print("RG1 first 3 dates:")
print(series1["date"].head(3).tolist())

print("\nRG0 last date -> RG1 first date gap:")
print(
    series1["date"].iloc[0] -
    series0["date"].iloc[-1]
)

del rg0
del rg1
del series0
del series1

import gc
gc.collect()

Test series: ('HOBBIES_1_001', 'CA_1')

Row group 0:
Rows: 1049
Date range: 2011-01-29 00:00:00 to 2013-12-12 00:00:00

Row group 1:
Rows: 864
Date range: 2013-12-13 00:00:00 to 2016-04-24 00:00:00

Boundary dates:
RG0 last 3 dates:
[Timestamp('2013-12-10 00:00:00'), Timestamp('2013-12-11 00:00:00'), Timestamp('2013-12-12 00:00:00')]
RG1 first 3 dates:
[Timestamp('2013-12-13 00:00:00'), Timestamp('2013-12-14 00:00:00'), Timestamp('2013-12-15 00:00:00')]

RG0 last date -> RG1 first date gap:
1 days 00:00:00


0

In [31]:
class StreamingBatchGenerator:

    def __init__(
        self,
        parquet_path,
        feature_columns,
        target_column,
        scaler,
        sequence_length,
        start_date,
        end_date,
        batch_size=64
    ):
        self.parquet_path = parquet_path
        self.feature_columns = feature_columns
        self.target_column = target_column
        self.scaler = scaler
        self.sequence_length = sequence_length
        self.start_date = pd.Timestamp(start_date)
        self.end_date = pd.Timestamp(end_date)
        self.batch_size = batch_size

        self.columns = list(
            dict.fromkeys(
                feature_columns
                + [
                    target_column,
                    "date",
                    "item_id",
                    "store_id"
                ]
            )
        )

    def batches(self):

        pf = pq.ParquetFile(self.parquet_path)

        # Keep last 28 rows for each active series
        history_buffer = {}

        X_batch = []
        y_batch = []

        for rg_idx in range(pf.num_row_groups):

            table = pf.read_row_group(
                rg_idx,
                columns=self.columns
            )

            df_rg = table.to_pandas()

            df_rg = df_rg.sort_values(
                ["item_id", "store_id", "date"]
            )

            for (item_id, store_id), group in df_rg.groupby(
                ["item_id", "store_id"],
                sort=False
            ):

                key = (item_id, store_id)

                group = group.reset_index(drop=True)

                # Add previous 28 days if this series
                # appeared in an earlier row group
                if key in history_buffer:

                    previous_history = history_buffer[key]

                    group = pd.concat(
                        [
                            previous_history,
                            group
                        ],
                        ignore_index=True
                    )

                # Prepare model features
                X_raw = prepare_features(
                    group,
                    self.feature_columns
                )

                X_scaled = self.scaler.transform(X_raw)

                y = group[
                    self.target_column
                ].to_numpy(dtype=np.float32)

                dates = group["date"].to_numpy()

                # Generate target positions
                for i in range(
                    self.sequence_length,
                    len(group)
                ):

                    target_date = dates[i]

                    if (
                        target_date >= self.start_date
                        and
                        target_date <= self.end_date
                    ):

                        X_batch.append(
                            X_scaled[
                                i-self.sequence_length:i
                            ]
                        )

                        y_batch.append(y[i])

                        if len(X_batch) == self.batch_size:

                            yield (
                                np.asarray(
                                    X_batch,
                                    dtype=np.float32
                                ),
                                np.asarray(
                                    y_batch,
                                    dtype=np.float32
                                )
                            )

                            X_batch = []
                            y_batch = []

                # Keep only last 28 rows as history
                history_buffer[key] = group.tail(
                    self.sequence_length
                ).copy()

            del df_rg
            del table

        # Final partial batch
        if len(X_batch) > 0:

            yield (
                np.asarray(
                    X_batch,
                    dtype=np.float32
                ),
                np.asarray(
                    y_batch,
                    dtype=np.float32
                )
            )


print("Cross-row-group streaming batch generator created successfully.")

Cross-row-group streaming batch generator created successfully.


In [32]:
# Test continuity of the same series across RG0 -> RG1

TEST_ITEM = "HOBBIES_1_001"
TEST_STORE = "CA_1"

test_columns = [
    "item_id",
    "store_id",
    "date",
    "sales"
]

rg0 = parquet_file.read_row_group(
    0,
    columns=test_columns
).to_pandas()

rg1 = parquet_file.read_row_group(
    1,
    columns=test_columns
).to_pandas()

series_rg0 = rg0[
    (rg0["item_id"] == TEST_ITEM) &
    (rg0["store_id"] == TEST_STORE)
].sort_values("date").reset_index(drop=True)

series_rg1 = rg1[
    (rg1["item_id"] == TEST_ITEM) &
    (rg1["store_id"] == TEST_STORE)
].sort_values("date").reset_index(drop=True)

# Last 28 days from RG0
history = series_rg0.tail(SEQUENCE_LENGTH)

# First rows from RG1
next_rows = series_rg1.head(5)

print("Test series:", TEST_ITEM, "/", TEST_STORE)

print("\nHistory buffer:")
print("Rows:", len(history))
print("Start:", history["date"].iloc[0])
print("End:", history["date"].iloc[-1])

print("\nNext row-group:")
print("First 5 dates:")
print(next_rows["date"].tolist())

print("\nBoundary check:")
print(
    "History last date:",
    history["date"].iloc[-1]
)
print(
    "Next date:",
    next_rows["date"].iloc[0]
)
print(
    "Gap:",
    next_rows["date"].iloc[0] -
    history["date"].iloc[-1]
)

print(
    "\nExpected history length:",
    SEQUENCE_LENGTH
)

print(
    "History length correct:",
    len(history) == SEQUENCE_LENGTH
)

del rg0
del rg1
del series_rg0
del series_rg1
del history
del next_rows

import gc
gc.collect()

Test series: HOBBIES_1_001 / CA_1

History buffer:
Rows: 28
Start: 2013-11-15 00:00:00
End: 2013-12-12 00:00:00

Next row-group:
First 5 dates:
[Timestamp('2013-12-13 00:00:00'), Timestamp('2013-12-14 00:00:00'), Timestamp('2013-12-15 00:00:00'), Timestamp('2013-12-16 00:00:00'), Timestamp('2013-12-17 00:00:00')]

Boundary check:
History last date: 2013-12-12 00:00:00
Next date: 2013-12-13 00:00:00
Gap: 1 days 00:00:00

Expected history length: 28
History length correct: True


0

In [33]:
def train_batch_stream():
    generator = StreamingBatchGenerator(
        parquet_path=DATA_PATH,
        feature_columns=FEATURE_COLUMNS,
        target_column=TARGET_COLUMN,
        scaler=scaler,
        sequence_length=SEQUENCE_LENGTH,
        start_date="2011-01-29",
        end_date=TRAIN_END_DATE,
        batch_size=BATCH_SIZE
    )

    yield from generator.batches()


def validation_batch_stream():
    generator = StreamingBatchGenerator(
        parquet_path=DATA_PATH,
        feature_columns=FEATURE_COLUMNS,
        target_column=TARGET_COLUMN,
        scaler=scaler,
        sequence_length=SEQUENCE_LENGTH,
        start_date=VALIDATION_START_DATE,
        end_date=VALIDATION_END_DATE,
        batch_size=BATCH_SIZE
    )

    yield from generator.batches()


output_signature = (
    tf.TensorSpec(
        shape=(None, SEQUENCE_LENGTH, len(FEATURE_COLUMNS)),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(None,),
        dtype=tf.float32
    )
)

train_dataset = tf.data.Dataset.from_generator(
    train_batch_stream,
    output_signature=output_signature
)

validation_dataset = tf.data.Dataset.from_generator(
    validation_batch_stream,
    output_signature=output_signature
)

print("tf.data datasets created successfully.")

tf.data datasets created successfully.


I0000 00:00:1789496529.470100     182 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789496529.473004     182 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [34]:
# Pull exactly one batch from the training dataset

X_tf_test, y_tf_test = next(iter(train_dataset))

print("TensorFlow training batch:")
print("X shape:", X_tf_test.shape)
print("y shape:", y_tf_test.shape)

print("\nDtypes:")
print("X dtype:", X_tf_test.dtype)
print("y dtype:", y_tf_test.dtype)

print("\nData quality:")
print(
    "NaN in X:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(X_tf_test), tf.int32)
    ).numpy()
)

print(
    "Inf in X:",
    tf.reduce_sum(
        tf.cast(tf.math.is_inf(X_tf_test), tf.int32)
    ).numpy()
)

print(
    "NaN in y:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(y_tf_test), tf.int32)
    ).numpy()
)

print(
    "Inf in y:",
    tf.reduce_sum(
        tf.cast(tf.math.is_inf(y_tf_test), tf.int32)
    ).numpy()
)

print("\nFirst 10 targets:")
print(y_tf_test[:10].numpy())

TensorFlow training batch:
X shape: (64, 28, 22)
y shape: (64,)

Dtypes:
X dtype: <dtype: 'float32'>
y dtype: <dtype: 'float32'>

Data quality:
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

First 10 targets:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [35]:
# Pull exactly one batch from the validation dataset

X_val_test, y_val_test = next(iter(validation_dataset))

print("TensorFlow validation batch:")
print("X shape:", X_val_test.shape)
print("y shape:", y_val_test.shape)

print("\nDtypes:")
print("X dtype:", X_val_test.dtype)
print("y dtype:", y_val_test.dtype)

print("\nData quality:")
print(
    "NaN in X:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(X_val_test), tf.int32)
    ).numpy()
)

print(
    "Inf in X:",
    tf.reduce_sum(
        tf.cast(tf.math.is_inf(X_val_test), tf.int32)
    ).numpy()
)

print(
    "NaN in y:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(y_val_test), tf.int32)
    ).numpy()
)

print(
    "Inf in y:",
    tf.reduce_sum(
        tf.cast(tf.math.is_inf(y_val_test), tf.int32)
    ).numpy()
)

print("\nFirst 10 validation targets:")
print(y_val_test[:10].numpy())

TensorFlow validation batch:
X shape: (64, 28, 22)
y shape: (64,)

Dtypes:
X dtype: <dtype: 'float32'>
y dtype: <dtype: 'float32'>

Data quality:
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [36]:
# GRU model

model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(SEQUENCE_LENGTH, len(FEATURE_COLUMNS))
    ),
    
    tf.keras.layers.GRU(
        64,
        activation="tanh"
    ),
    
    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),
    
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,009 (74.25 KB)

 Trainable params: 19,009 (74.25 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
# Train GRU model

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1j
)

Epoch 1/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 52s 10ms/step - loss: 6.6564 - mae: 0.8717 - val_loss: 4.1690 - val_mae: 1.0104
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 50s 10ms/step - loss: 3.5108 - mae: 0.5613 - val_loss: 4.0391 - val_mae: 0.9658
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 50s 10ms/step - loss: 1.5976 - mae: 0.6469 - val_loss: 4.2085 - val_mae: 1.0655
Epoch 4/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 52s 10ms/step - loss: 4.6767 - mae: 0.9388 - val_loss: 4.0141 - val_mae: 0.9952
Epoch 5/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 53s 11ms/step - loss: 3.3753 - mae: 0.7972 - val_loss: 3.8383 - val_mae: 0.9926
Epoch 6/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 51s 10ms/step - loss: 3.0620 - mae: 0.9691 - val_loss: 3.9897 - val_mae: 0.9650
Epoch 7/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 50s 10ms/step - loss: 2.9488 - mae: 0.8067 - val_loss: 3.9625 - val_mae: 0.9541


In [38]:
# Generate validation predictions efficiently

val_predictions = model.predict(
    validation_dataset,
    steps=VALIDATION_STEPS,
    verbose=1
).reshape(-1)

print("Prediction shape:", val_predictions.shape)
print("Expected predictions:", VALIDATION_STEPS * BATCH_SIZE)

print("NaN predictions:", np.isnan(val_predictions).sum())
print("Inf predictions:", np.isinf(val_predictions).sum())

print("\nFirst 10 predictions:")
print(val_predictions[:10])

500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 31ms/step
Prediction shape: (32000,)
Expected predictions: 32000
NaN predictions: 0
Inf predictions: 0

First 10 predictions:
[1.0014819  1.0138174  0.9301846  0.9288162  1.0101554  1.412784
 1.2363341  0.79979247 0.5992918  0.8860817 ]


In [39]:
# Generate predictions for the complete validation period

val_predictions = model.predict(
    validation_dataset,
    verbose=1
).reshape(-1)

print("Prediction shape:", val_predictions.shape)

print("NaN predictions:", np.isnan(val_predictions).sum())
print("Inf predictions:", np.isinf(val_predictions).sum())

print("\nFirst 10 predictions:")
print(val_predictions[:10])

13340/13340 ━━━━━━━━━━━━━━━━━━━━ 436s 32ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Prediction shape: (853720,)
NaN predictions: 0
Inf predictions: 0

First 10 predictions:
[1.0014819  1.0138174  0.9301846  0.9288162  1.0101554  1.412784
 1.2363341  0.79979247 0.5992918  0.8860817 ]


In [40]:
# Collect actual validation targets

val_actuals = []

for _, y_batch in validation_dataset:
    val_actuals.append(y_batch.numpy())

val_actuals = np.concatenate(val_actuals)

print("Actuals shape:", val_actuals.shape)

print("NaN actuals:", np.isnan(val_actuals).sum())
print("Inf actuals:", np.isinf(val_actuals).sum())

print("\nFirst 10 actual values:")
print(val_actuals[:10])

Actuals shape: (853720,)
NaN actuals: 0
Inf actuals: 0

First 10 actual values:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [41]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate metrics
mae = mean_absolute_error(val_actuals, val_predictions)

rmse = np.sqrt(
    mean_squared_error(val_actuals, val_predictions)
)

wape = (
    np.sum(np.abs(val_actuals - val_predictions))
    / np.sum(np.abs(val_actuals))
) * 100

print("GRU Validation Metrics")
print("=" * 30)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"WAPE : {wape:.4f}%")

GRU Validation Metrics
MAE  : 1.0691
RMSE : 2.5656
WAPE : 77.1137%


In [42]:
best_epoch = np.argmin(history.history["val_loss"]) + 1
best_val_loss = np.min(history.history["val_loss"])
best_val_mae = history.history["val_mae"][best_epoch - 1]

print("GRU Training Summary")
print("=" * 30)
print(f"Best epoch      : {best_epoch}")
print(f"Best val_loss   : {best_val_loss:.4f}")
print(f"Val MAE at best : {best_val_mae:.4f}")
print(f"Total epochs    : {len(history.history['loss'])}")

GRU Training Summary
Best epoch      : 5
Best val_loss   : 3.8383
Val MAE at best : 0.9926
Total epochs    : 7


In [43]:
best_epoch = np.argmin(history.history["val_loss"]) + 1

best_val_loss = history.history["val_loss"][best_epoch - 1]
best_val_mae = history.history["val_mae"][best_epoch - 1]

print("GRU Training Summary")
print("=" * 30)
print(f"Best epoch      : {best_epoch}")
print(f"Best val_loss   : {best_val_loss:.4f}")
print(f"Val MAE at best : {best_val_mae:.4f}")
print(f"Total epochs    : {len(history.history['loss'])}")

GRU Training Summary
Best epoch      : 5
Best val_loss   : 3.8383
Val MAE at best : 0.9926
Total epochs    : 7


# GRU Model — Final Results

## Model Configuration

- **Sequence Length:** 28 days
- **Forecast Horizon:** 1 day
- **Input Features:** 22
- **GRU Units:** 64
- **Dense Layer:** 32 neurons with ReLU activation
- **Output Layer:** 1 neuron
- **Optimizer:** Adam
- **Learning Rate:** 0.001
- **Loss Function:** Mean Squared Error (MSE)
- **Training Batch Size:** 64
- **Maximum Epochs:** 10
- **Early Stopping:** Patience = 2
- **Restore Best Weights:** True

## Training Results

Training stopped after **7 epochs** because validation loss did not improve for two consecutive epochs.

- **Best Epoch:** 5
- **Best Validation Loss:** 3.8383
- **Validation MAE at Best Epoch:** 0.9926
- **Total Epochs Completed:** 7
- **Total Trainable Parameters:** 19,009

The best weights from epoch 5 were restored using early stopping.

## Validation Results

The model was evaluated on the complete validation period:

**2016-03-28 to 2016-04-24**

- **Validation Observations:** 853,720
- **MAE:** 1.0691
- **RMSE:** 2.5656
- **WAPE:** 77.1137%

No NaN or Inf values were found in the validation predictions.

## Comparison with Previous Models

| Model | MAE | RMSE | WAPE |
|---|---:|---:|---:|
| 28-day Moving Average | 1.0050 | 2.0935 | 72.4875% |
| LSTM | 1.0220 | 2.3766 | 73.7131% |
| RNN | 1.0326 | 2.4927 | 74.4774% |
| GRU | 1.0691 | 2.5656 | 77.1137% |
| Naive | 1.1798 | 2.5948 | 85.0972% |
| Seasonal Naive | 1.2054 | 2.6601 | 86.9390% |

## Conclusion

The GRU model successfully learned the forecasting task and produced valid predictions for the complete validation set. However, under the same experimental setup, the GRU performed worse than both the RNN and LSTM models across MAE, RMSE, and WAPE.

The **28-day Moving Average remained the best-performing approach**, achieving the lowest MAE, RMSE, and WAPE.

Among the recurrent neural network models, the performance ranking was:

**LSTM > RNN > GRU**

The GRU used fewer parameters than the LSTM and trained efficiently on the Kaggle GPU, but this computational efficiency did not translate into better forecasting accuracy in this experiment.

> **Stage Status: COMPLETE — GRU experiment finalized.**